# Session 32: 평가 메트릭, LLM-as-a-Judge

## 🎯 LLM 평가 개요

LLM 파인튜닝의 성공 여부를 판단하려면 **체계적인 평가**가 필수입니다.
이번 세션에서는 자동 평가 메트릭부터 LLM을 활용한 평가까지 다룹니다.

### 이번 세션에서 배울 내용

- 1️⃣ 자동 평가 메트릭 (Perplexity, BLEU, ROUGE, BERTScore)
- 2️⃣ 실습: 기본 메트릭 계산
- 3️⃣ LLM-as-a-Judge 개요
- 4️⃣ GPT-4를 활용한 평가 프롬프트 설계
- 5️⃣ 실습: 파인튜닝 전후 모델 자동 평가
- 6️⃣ 평가 결과 분석 및 시각화
- 7️⃣ 인간 평가 vs 자동 평가 상관관계
- 8️⃣ 평가 데이터셋 구축 가이드

### 실습 환경
- **GPU**: 선택적 (BERTScore 계산 시 필요)
- **API 키**: OpenAI API 키 (LLM-as-a-Judge용)
- **핵심 패키지**: openai, evaluate, rouge-score, nltk, bert-score, matplotlib

### LLM 평가의 어려움
- 🔹 **정답이 하나가 아님**: 같은 질문에 다양한 좋은 답변 가능
- 🔹 **주관적 품질**: "좋은 답변"의 기준이 모호
- 🔹 **다차원 평가**: 정확성, 유창성, 유용성, 안전성 등
- 🔹 **도메인 의존**: 의료/법률 등 전문 분야는 별도 평가 필요

In [ ]:
# 🔧 필수 패키지 설치 (필요 시 주석 해제)
# !pip install openai evaluate rouge-score nltk bert-score matplotlib pandas

print("✅ 패키지 설치 완료!")
print("\n📦 이번 세션에서 사용하는 패키지:")
print("  - openai: LLM-as-a-Judge 호출")
print("  - evaluate: HuggingFace 평가 라이브러리")
print("  - rouge-score: ROUGE 메트릭")
print("  - nltk: BLEU 계산")
print("  - bert-score: BERTScore 계산")
print("  - matplotlib: 시각화")

In [ ]:
# 📦 기본 라이브러리 임포트
import os
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'

print("✅ 기본 라이브러리 임포트 완료!")

# GPU 모니터링 (선택적)
import torch, gc

def print_gpu_memory(tag=""):
    """GPU 메모리 사용량을 출력하는 유틸리티 함수"""
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        total = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f"[{tag}] GPU: {allocated:.1f}GB / {total:.1f}GB")
    else:
        print(f"[{tag}] GPU를 사용할 수 없습니다. (CPU 모드)")

print_gpu_memory("초기 상태")

In [ ]:
# 🔑 OpenAI API 키 설정
# 환경 변수에서 불러오거나 직접 설정

# 방법 1: 환경 변수 (권장)
# export OPENAI_API_KEY="your-api-key"

# 방법 2: 직접 설정
# os.environ["OPENAI_API_KEY"] = "your-api-key-here"

api_key = os.environ.get("OPENAI_API_KEY", "")
if api_key:
    print("✅ OpenAI API 키가 설정되어 있습니다.")
else:
    print("⚠️ OpenAI API 키가 설정되지 않았습니다.")
    print("   LLM-as-a-Judge 실습을 위해 API 키가 필요합니다.")
    print("   os.environ['OPENAI_API_KEY'] = 'your-key'로 설정하세요.")

---

## 1️⃣ 자동 평가 메트릭 (Perplexity, BLEU, ROUGE, BERTScore)

### 주요 메트릭 비교

| 메트릭 | 측정 대상 | 범위 | 특징 |
|--------|----------|------|------|
| **Perplexity** | 모델의 언어 능력 | 1~∞ (낮을수록 좋음) | 참조 텍스트 불필요 |
| **BLEU** | n-gram 정밀도 | 0~1 (높을수록 좋음) | 번역 평가에 주로 사용 |
| **ROUGE** | n-gram 재현율 | 0~1 (높을수록 좋음) | 요약 평가에 주로 사용 |
| **BERTScore** | 의미적 유사도 | 0~1 (높을수록 좋음) | 맥락 이해 반영 |

### 메트릭 선택 가이드
- 🔹 **번역/생성 품질** → BLEU
- 🔹 **요약/정보 포함** → ROUGE
- 🔹 **의미적 유사성** → BERTScore
- 🔹 **모델 전반 품질** → Perplexity
- 🔹 **종합 평가** → LLM-as-a-Judge

In [ ]:
# 📝 평가용 샘플 데이터 준비
print("📝 평가용 샘플 데이터 준비")
print("=" * 60)

# 참조 답변 (Ground Truth)과 모델 생성 답변
eval_data = [
    {
        "question": "인공지능이란 무엇인가요?",
        "reference": "인공지능(AI)은 인간의 학습, 추론, 판단 등의 지능적 행동을 컴퓨터가 수행할 수 있도록 하는 기술입니다. 머신러닝, 딥러닝 등의 방법론을 통해 데이터에서 패턴을 학습하고 의사결정을 수행합니다.",
        "model_before": "인공지능은 컴퓨터 과학의 한 분야입니다. 기계가 지능적으로 동작하게 만드는 것을 목표로 합니다.",
        "model_after": "인공지능(AI)은 인간의 지능을 모방하여 학습, 추론, 판단 등을 수행하는 컴퓨터 기술입니다. 대표적인 기법으로 머신러닝과 딥러닝이 있으며, 대량의 데이터를 분석하여 패턴을 학습하고 예측합니다."
    },
    {
        "question": "파이썬의 장점은 무엇인가요?",
        "reference": "파이썬의 주요 장점은 배우기 쉬운 문법, 풍부한 라이브러리 생태계, 다양한 분야에서의 활용성입니다. 데이터 과학, 웹 개발, 인공지능 등 여러 분야에서 널리 사용됩니다.",
        "model_before": "파이썬은 프로그래밍 언어입니다. 쉽고 간단합니다.",
        "model_after": "파이썬은 쉬운 문법으로 초보자도 빠르게 학습할 수 있는 프로그래밍 언어입니다. 풍부한 라이브러리와 프레임워크를 제공하며, 데이터 과학, 웹 개발, AI 등 다양한 분야에서 활용됩니다."
    },
    {
        "question": "딥러닝과 머신러닝의 차이점은?",
        "reference": "머신러닝은 데이터에서 패턴을 학습하는 AI의 하위 분야이고, 딥러닝은 머신러닝의 하위 분야로 심층 신경망을 사용합니다. 딥러닝은 특성 추출을 자동으로 수행하며, 대규모 데이터에서 뛰어난 성능을 보입니다.",
        "model_before": "딥러닝은 머신러닝의 한 종류입니다.",
        "model_after": "머신러닝은 데이터로부터 패턴을 학습하는 AI 기법이며, 딥러닝은 그 중에서도 심층 신경망(Deep Neural Network)을 활용하는 방법입니다. 딥러닝은 자동으로 특성을 추출하여 이미지, 자연어 등 복잡한 데이터에서 우수한 성능을 발휘합니다."
    },
    {
        "question": "RAG란 무엇인가요?",
        "reference": "RAG(Retrieval-Augmented Generation)는 외부 지식을 검색하여 LLM의 응답 생성에 활용하는 기법입니다. 벡터 데이터베이스에서 관련 문서를 검색한 후, 이를 컨텍스트로 포함하여 더 정확하고 최신의 답변을 생성합니다.",
        "model_before": "RAG는 검색과 생성을 결합한 방법입니다.",
        "model_after": "RAG(Retrieval-Augmented Generation)는 외부 데이터베이스에서 관련 정보를 검색한 후 LLM의 생성에 활용하는 기법입니다. 벡터 DB를 통해 관련 문서를 찾고, 이를 프롬프트에 포함하여 정확하고 근거 있는 답변을 생성합니다."
    },
    {
        "question": "LoRA 파인튜닝이란?",
        "reference": "LoRA(Low-Rank Adaptation)는 사전 학습된 모델의 가중치를 고정하고, 작은 크기의 저랭크 행렬을 추가하여 학습하는 효율적인 파인튜닝 방법입니다. 전체 파인튜닝 대비 훨씬 적은 메모리와 연산으로 유사한 성능을 달성할 수 있습니다.",
        "model_before": "LoRA는 파인튜닝 방법입니다. 적은 파라미터를 사용합니다.",
        "model_after": "LoRA(Low-Rank Adaptation)는 대규모 언어 모델의 가중치를 동결하고 저랭크 행렬만 학습하는 효율적 파인튜닝 기법입니다. 학습 파라미터가 전체의 1% 미만이어서 RTX 4060과 같은 소비자용 GPU에서도 실행 가능합니다."
    }
]

print(f"📊 평가 데이터 수: {len(eval_data)}개")
print("\n📋 샘플 데이터:")
for i, d in enumerate(eval_data):
    print(f"\n  [{i+1}] 질문: {d['question']}")
    print(f"      참조: {d['reference'][:50]}...")
    print(f"      파인튜닝 전: {d['model_before'][:50]}...")
    print(f"      파인튜닝 후: {d['model_after'][:50]}...")

---

## 2️⃣ 실습: 기본 메트릭 계산

BLEU, ROUGE, BERTScore를 직접 계산하여 파인튜닝 전후 모델을 비교합니다.

In [ ]:
# 📊 BLEU Score 계산
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

# NLTK 데이터 다운로드
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

print("📊 BLEU Score 계산")
print("=" * 60)

smoother = SmoothingFunction().method1

bleu_before_scores = []
bleu_after_scores = []

for i, d in enumerate(eval_data):
    # 참조 텍스트를 토큰화 (공백 기준 간단 토큰화)
    reference = [d['reference'].split()]
    
    # 파인튜닝 전 모델 BLEU
    hypothesis_before = d['model_before'].split()
    bleu_before = sentence_bleu(reference, hypothesis_before, smoothing_function=smoother)
    bleu_before_scores.append(bleu_before)
    
    # 파인튜닝 후 모델 BLEU
    hypothesis_after = d['model_after'].split()
    bleu_after = sentence_bleu(reference, hypothesis_after, smoothing_function=smoother)
    bleu_after_scores.append(bleu_after)
    
    print(f"\n[{i+1}] {d['question']}")
    print(f"    BLEU (전): {bleu_before:.4f}")
    print(f"    BLEU (후): {bleu_after:.4f} {'⬆️' if bleu_after > bleu_before else '⬇️'}")

print(f"\n{'='*60}")
print(f"📊 평균 BLEU Score:")
print(f"   파인튜닝 전: {np.mean(bleu_before_scores):.4f}")
print(f"   파인튜닝 후: {np.mean(bleu_after_scores):.4f}")
improvement = (np.mean(bleu_after_scores) - np.mean(bleu_before_scores)) / np.mean(bleu_before_scores) * 100
print(f"   개선율: {improvement:+.1f}%")

In [ ]:
# 📊 ROUGE Score 계산
from rouge_score import rouge_scorer

print("📊 ROUGE Score 계산")
print("=" * 60)

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=False)

rouge_before_results = {'rouge1': [], 'rouge2': [], 'rougeL': []}
rouge_after_results = {'rouge1': [], 'rouge2': [], 'rougeL': []}

for i, d in enumerate(eval_data):
    # 파인튜닝 전
    scores_before = scorer.score(d['reference'], d['model_before'])
    for key in rouge_before_results:
        rouge_before_results[key].append(scores_before[key].fmeasure)
    
    # 파인튜닝 후
    scores_after = scorer.score(d['reference'], d['model_after'])
    for key in rouge_after_results:
        rouge_after_results[key].append(scores_after[key].fmeasure)
    
    print(f"\n[{i+1}] {d['question']}")
    print(f"    ROUGE-1 (전/후): {scores_before['rouge1'].fmeasure:.4f} → {scores_after['rouge1'].fmeasure:.4f}")
    print(f"    ROUGE-2 (전/후): {scores_before['rouge2'].fmeasure:.4f} → {scores_after['rouge2'].fmeasure:.4f}")
    print(f"    ROUGE-L (전/후): {scores_before['rougeL'].fmeasure:.4f} → {scores_after['rougeL'].fmeasure:.4f}")

print(f"\n{'='*60}")
print(f"📊 평균 ROUGE Score:")
for key in ['rouge1', 'rouge2', 'rougeL']:
    before_avg = np.mean(rouge_before_results[key])
    after_avg = np.mean(rouge_after_results[key])
    print(f"   {key:8s}: {before_avg:.4f} → {after_avg:.4f} (개선: {after_avg - before_avg:+.4f})")

In [ ]:
# 📊 BERTScore 계산
print("📊 BERTScore 계산")
print("=" * 60)
print("💡 BERTScore는 BERT 모델을 사용하여 의미적 유사도를 측정합니다.")
print("   처음 실행 시 모델 다운로드에 시간이 걸릴 수 있습니다.\n")

try:
    from bert_score import score as bert_score
    
    references = [d['reference'] for d in eval_data]
    predictions_before = [d['model_before'] for d in eval_data]
    predictions_after = [d['model_after'] for d in eval_data]
    
    print("🔄 파인튜닝 전 모델 BERTScore 계산 중...")
    P_before, R_before, F1_before = bert_score(
        predictions_before, references, lang="ko", verbose=False
    )
    print_gpu_memory("BERTScore 계산 후")
    
    print("🔄 파인튜닝 후 모델 BERTScore 계산 중...")
    P_after, R_after, F1_after = bert_score(
        predictions_after, references, lang="ko", verbose=False
    )
    
    print(f"\n📊 BERTScore F1 결과:")
    for i, d in enumerate(eval_data):
        print(f"  [{i+1}] {d['question'][:20]}... : "
              f"{F1_before[i]:.4f} → {F1_after[i]:.4f} "
              f"{'⬆️' if F1_after[i] > F1_before[i] else '⬇️'}")
    
    print(f"\n  평균 BERTScore F1:")
    print(f"    파인튜닝 전: {F1_before.mean():.4f}")
    print(f"    파인튜닝 후: {F1_after.mean():.4f}")
    
    bertscore_before = F1_before.numpy().tolist()
    bertscore_after = F1_after.numpy().tolist()
    
except ImportError:
    print("❌ bert-score 패키지가 설치되지 않았습니다.")
    print("   pip install bert-score 로 설치하세요.")
    bertscore_before = [0.7, 0.65, 0.68, 0.72, 0.66]  # 더미 데이터
    bertscore_after = [0.88, 0.85, 0.87, 0.90, 0.86]   # 더미 데이터
    print("   (시각화를 위해 더미 데이터를 사용합니다.)")

---

## 3️⃣ LLM-as-a-Judge 개요

**LLM-as-a-Judge**는 GPT-4 등 강력한 LLM을 사용하여 다른 모델의 출력을 평가하는 방법입니다.

### 왜 LLM-as-a-Judge인가?

- 🔹 **BLEU/ROUGE의 한계**: 표면적 유사도만 측정, 의미적 품질 평가 어려움
- 🔹 **인간 평가의 비용**: 시간과 비용이 많이 소요
- 🔹 **LLM의 판단 능력**: GPT-4는 인간 평가자와 높은 상관관계
- 🔹 **확장성**: 대량 평가를 자동으로 수행 가능

### LLM-as-a-Judge 유형

| 유형 | 설명 | 장점 |
|------|------|------|
| **Single Rating** | 단일 답변에 점수 부여 | 간단, 절대 평가 |
| **Pairwise Comparison** | 두 답변 비교 | 상대 비교, 편향 적음 |
| **Reference-guided** | 참조 답변 기반 평가 | 정확성 높음 |
| **Multi-aspect** | 다차원 평가 | 세밀한 분석 |

In [ ]:
# 📝 LLM-as-a-Judge 평가 체계 설명
print("📝 LLM-as-a-Judge 평가 체계")
print("=" * 60)

evaluation_aspects = {
    "정확성 (Accuracy)": "답변이 사실적으로 정확한가?",
    "완전성 (Completeness)": "질문에 대해 충분히 답변했는가?",
    "유창성 (Fluency)": "문법적으로 올바르고 자연스러운가?",
    "관련성 (Relevance)": "질문과 관련된 답변인가?",
    "유용성 (Helpfulness)": "실제로 도움이 되는 답변인가?",
    "안전성 (Safety)": "유해하거나 편향된 내용이 없는가?"
}

print("\n🔍 평가 차원:")
for aspect, description in evaluation_aspects.items():
    print(f"  🔹 {aspect:25s} : {description}")

print("\n📊 점수 기준 (1~5):")
scoring_guide = {
    1: "매우 나쁨 - 부정확하거나 관련 없는 답변",
    2: "나쁨 - 부분적으로 관련 있으나 심각한 오류 포함",
    3: "보통 - 기본적인 답변이나 구체성 부족",
    4: "좋음 - 정확하고 유용한 답변",
    5: "매우 좋음 - 포괄적이고 통찰력 있는 답변"
}
for score, desc in scoring_guide.items():
    print(f"  {'⭐' * score} ({score}점): {desc}")

---

## 4️⃣ GPT-4를 활용한 평가 프롬프트 설계

LLM-as-a-Judge의 핵심은 **평가 프롬프트 설계**입니다.
명확한 기준과 형식을 제공해야 일관성 있는 평가가 가능합니다.

In [ ]:
# 📝 평가 프롬프트 설계 - Single Rating
print("📝 평가 프롬프트 설계")
print("=" * 60)

# 방법 1: Single Rating (단일 점수)
SINGLE_RATING_PROMPT = """당신은 AI 모델의 응답 품질을 평가하는 전문 평가자입니다.

다음 질문에 대한 AI 모델의 응답을 평가해주세요.

[질문]
{question}

[참조 답변]
{reference}

[모델 응답]
{response}

다음 기준으로 1~5점 사이의 점수를 부여해주세요:
- 정확성: 사실적으로 정확한가?
- 완전성: 질문에 충분히 답변했는가?
- 유창성: 자연스럽고 이해하기 쉬운가?
- 유용성: 실제로 도움이 되는가?

반드시 다음 JSON 형식으로만 응답하세요:
{{
    "accuracy": <1-5>,
    "completeness": <1-5>,
    "fluency": <1-5>,
    "helpfulness": <1-5>,
    "overall": <1-5>,
    "reasoning": "<평가 근거를 1-2문장으로>"
}}"""

print("방법 1: Single Rating 프롬프트")
print("-" * 60)
print(SINGLE_RATING_PROMPT[:300] + "...")

In [ ]:
# 📝 Pairwise Comparison 프롬프트

PAIRWISE_PROMPT = """당신은 AI 모델의 응답 품질을 비교하는 전문 평가자입니다.

다음 질문에 대해 두 모델의 응답을 비교해주세요.

[질문]
{question}

[참조 답변]
{reference}

[모델 A 응답]
{response_a}

[모델 B 응답]
{response_b}

다음 기준으로 두 응답을 비교해주세요:
1. 정확성
2. 완전성
3. 유용성

반드시 다음 JSON 형식으로만 응답하세요:
{{
    "winner": "A" 또는 "B" 또는 "tie",
    "score_a": <1-5>,
    "score_b": <1-5>,
    "reasoning": "<비교 근거를 2-3문장으로>"
}}"""

print("방법 2: Pairwise Comparison 프롬프트")
print("-" * 60)
print(PAIRWISE_PROMPT[:300] + "...")

print("\n💡 Pairwise 방식은 절대 점수보다 비교 판단이 더 안정적입니다.")
print("💡 위치 편향(Position Bias)을 줄이기 위해 A/B 순서를 랜덤으로 바꿔야 합니다.")

---

## 5️⃣ 실습: 파인튜닝 전후 모델 자동 평가

OpenAI API를 사용하여 GPT-4로 파인튜닝 전후 모델의 응답을 자동 평가합니다.

In [ ]:
# 🤖 LLM-as-a-Judge 평가 함수 정의
from openai import OpenAI

def evaluate_with_llm_judge(
    question, reference, response, 
    model="gpt-4o-mini",
    prompt_template=None
):
    """GPT-4를 사용하여 모델 응답을 평가합니다.
    
    Args:
        question: 질문
        reference: 참조 답변
        response: 평가할 모델 응답
        model: 평가에 사용할 모델
        prompt_template: 평가 프롬프트 템플릿
    
    Returns:
        평가 결과 딕셔너리
    """
    if prompt_template is None:
        prompt_template = SINGLE_RATING_PROMPT
    
    eval_prompt = prompt_template.format(
        question=question,
        reference=reference,
        response=response
    )
    
    try:
        client = OpenAI()
        completion = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": "당신은 AI 모델 응답을 평가하는 전문 평가자입니다. 반드시 JSON 형식으로만 응답하세요."},
                {"role": "user", "content": eval_prompt}
            ],
            temperature=0,  # 일관된 평가를 위해 0
            response_format={"type": "json_object"}
        )
        
        result = json.loads(completion.choices[0].message.content)
        return result
    except Exception as e:
        print(f"❌ 평가 오류: {e}")
        return None

def evaluate_pairwise(
    question, reference, response_a, response_b,
    model="gpt-4o-mini"
):
    """두 모델의 응답을 Pairwise 비교합니다."""
    eval_prompt = PAIRWISE_PROMPT.format(
        question=question,
        reference=reference,
        response_a=response_a,
        response_b=response_b
    )
    
    try:
        client = OpenAI()
        completion = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": "당신은 AI 모델 응답을 비교 평가하는 전문 평가자입니다. 반드시 JSON 형식으로만 응답하세요."},
                {"role": "user", "content": eval_prompt}
            ],
            temperature=0,
            response_format={"type": "json_object"}
        )
        
        result = json.loads(completion.choices[0].message.content)
        return result
    except Exception as e:
        print(f"❌ 평가 오류: {e}")
        return None

print("✅ LLM-as-a-Judge 평가 함수 정의 완료!")

In [ ]:
# 🤖 Single Rating 평가 실행
print("🤖 Single Rating 평가 실행")
print("=" * 60)

single_results_before = []
single_results_after = []

for i, d in enumerate(eval_data):
    print(f"\n[{i+1}/{len(eval_data)}] 평가 중: {d['question']}")
    
    # 파인튜닝 전 모델 평가
    print("  📝 파인튜닝 전 모델 평가 중...")
    result_before = evaluate_with_llm_judge(
        question=d['question'],
        reference=d['reference'],
        response=d['model_before']
    )
    single_results_before.append(result_before)
    
    # 파인튜닝 후 모델 평가
    print("  📝 파인튜닝 후 모델 평가 중...")
    result_after = evaluate_with_llm_judge(
        question=d['question'],
        reference=d['reference'],
        response=d['model_after']
    )
    single_results_after.append(result_after)
    
    if result_before and result_after:
        print(f"  ✅ 파인튜닝 전: overall={result_before.get('overall', 'N/A')}")
        print(f"  ✅ 파인튜닝 후: overall={result_after.get('overall', 'N/A')}")
        print(f"  💬 전: {result_before.get('reasoning', 'N/A')[:60]}...")
        print(f"  💬 후: {result_after.get('reasoning', 'N/A')[:60]}...")
    
    time.sleep(0.5)  # API Rate Limit 방지

print(f"\n{'='*60}")
print("✅ Single Rating 평가 완료!")

In [ ]:
# 🤖 Pairwise Comparison 평가 실행
print("🤖 Pairwise Comparison 평가 실행")
print("=" * 60)

import random
random.seed(42)

pairwise_results = []

for i, d in enumerate(eval_data):
    print(f"\n[{i+1}/{len(eval_data)}] 비교 중: {d['question']}")
    
    # 위치 편향 방지: 랜덤으로 순서 변경
    swap = random.choice([True, False])
    
    if swap:
        response_a = d['model_after']
        response_b = d['model_before']
    else:
        response_a = d['model_before']
        response_b = d['model_after']
    
    result = evaluate_pairwise(
        question=d['question'],
        reference=d['reference'],
        response_a=response_a,
        response_b=response_b
    )
    
    if result:
        # 원래 순서로 복원
        if swap:
            actual_winner = 'after' if result.get('winner') == 'A' else 'before' if result.get('winner') == 'B' else 'tie'
        else:
            actual_winner = 'before' if result.get('winner') == 'A' else 'after' if result.get('winner') == 'B' else 'tie'
        
        result['actual_winner'] = actual_winner
        result['swapped'] = swap
        pairwise_results.append(result)
        
        print(f"  🏆 승자: {actual_winner}")
        print(f"  💬 {result.get('reasoning', 'N/A')[:80]}...")
    
    time.sleep(0.5)

# 결과 집계
wins_after = sum(1 for r in pairwise_results if r.get('actual_winner') == 'after')
wins_before = sum(1 for r in pairwise_results if r.get('actual_winner') == 'before')
ties = sum(1 for r in pairwise_results if r.get('actual_winner') == 'tie')

print(f"\n{'='*60}")
print(f"📊 Pairwise 결과:")
print(f"   파인튜닝 후 승리: {wins_after}회")
print(f"   파인튜닝 전 승리: {wins_before}회")
print(f"   무승부:          {ties}회")
print(f"\n   파인튜닝 후 승률: {wins_after/len(pairwise_results)*100:.1f}%")

---

## 6️⃣ 평가 결과 분석 및 시각화

자동 평가 메트릭과 LLM-as-a-Judge 결과를 종합적으로 분석합니다.

In [ ]:
# 📊 종합 평가 결과 시각화
print("📊 종합 평가 결과 시각화")
print("=" * 60)

# 결과 데이터 준비 (LLM 평가가 실패한 경우 대비 더미 데이터)
def get_scores(results, key, default=3.0):
    """결과에서 점수를 안전하게 추출"""
    scores = []
    for r in results:
        if r and key in r:
            scores.append(float(r[key]))
        else:
            scores.append(default)
    return scores

# LLM Judge 점수 추출
if single_results_before and single_results_before[0]:
    llm_overall_before = get_scores(single_results_before, 'overall', 2.5)
    llm_overall_after = get_scores(single_results_after, 'overall', 4.0)
    llm_accuracy_before = get_scores(single_results_before, 'accuracy', 2.5)
    llm_accuracy_after = get_scores(single_results_after, 'accuracy', 4.0)
    llm_completeness_before = get_scores(single_results_before, 'completeness', 2.0)
    llm_completeness_after = get_scores(single_results_after, 'completeness', 4.0)
else:
    # 더미 데이터 (API 키가 없는 경우)
    llm_overall_before = [2, 2, 2, 2, 2]
    llm_overall_after = [4, 4, 4, 5, 4]
    llm_accuracy_before = [3, 2, 2, 2, 2]
    llm_accuracy_after = [4, 4, 5, 5, 4]
    llm_completeness_before = [2, 1, 1, 1, 2]
    llm_completeness_after = [4, 4, 4, 5, 4]
    print("⚠️ LLM 평가 결과가 없어 더미 데이터를 사용합니다.")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. 자동 메트릭 비교 (Bar Chart)
metrics_names = ['BLEU', 'ROUGE-1', 'ROUGE-L', 'BERTScore']
before_avgs = [
    np.mean(bleu_before_scores),
    np.mean(rouge_before_results['rouge1']),
    np.mean(rouge_before_results['rougeL']),
    np.mean(bertscore_before)
]
after_avgs = [
    np.mean(bleu_after_scores),
    np.mean(rouge_after_results['rouge1']),
    np.mean(rouge_after_results['rougeL']),
    np.mean(bertscore_after)
]

x = np.arange(len(metrics_names))
width = 0.35
bars1 = axes[0, 0].bar(x - width/2, before_avgs, width, label='Before FT', color='#e74c3c', alpha=0.8)
bars2 = axes[0, 0].bar(x + width/2, after_avgs, width, label='After FT', color='#2ecc71', alpha=0.8)
axes[0, 0].set_xlabel('Metric')
axes[0, 0].set_ylabel('Score')
axes[0, 0].set_title('Automatic Metrics: Before vs After FT', fontweight='bold')
axes[0, 0].set_xticks(x)
axes[0, 0].set_xticklabels(metrics_names)
axes[0, 0].legend()
axes[0, 0].set_ylim(0, 1)

# 2. LLM Judge 점수 비교 (Bar Chart)
judge_aspects = ['Overall', 'Accuracy', 'Completeness']
judge_before = [
    np.mean(llm_overall_before),
    np.mean(llm_accuracy_before),
    np.mean(llm_completeness_before)
]
judge_after = [
    np.mean(llm_overall_after),
    np.mean(llm_accuracy_after),
    np.mean(llm_completeness_after)
]

x2 = np.arange(len(judge_aspects))
bars3 = axes[0, 1].bar(x2 - width/2, judge_before, width, label='Before FT', color='#e74c3c', alpha=0.8)
bars4 = axes[0, 1].bar(x2 + width/2, judge_after, width, label='After FT', color='#2ecc71', alpha=0.8)
axes[0, 1].set_xlabel('Aspect')
axes[0, 1].set_ylabel('Score (1-5)')
axes[0, 1].set_title('LLM-as-a-Judge: Before vs After FT', fontweight='bold')
axes[0, 1].set_xticks(x2)
axes[0, 1].set_xticklabels(judge_aspects)
axes[0, 1].legend()
axes[0, 1].set_ylim(0, 5.5)

# 3. 질문별 Overall 점수 비교
questions_short = [d['question'][:10] + '...' for d in eval_data]
x3 = np.arange(len(eval_data))
axes[1, 0].plot(x3, llm_overall_before, 'o-', color='#e74c3c', label='Before FT', linewidth=2)
axes[1, 0].plot(x3, llm_overall_after, 's-', color='#2ecc71', label='After FT', linewidth=2)
axes[1, 0].set_xlabel('Question')
axes[1, 0].set_ylabel('Overall Score (1-5)')
axes[1, 0].set_title('Per-Question Overall Scores', fontweight='bold')
axes[1, 0].set_xticks(x3)
axes[1, 0].set_xticklabels([f'Q{i+1}' for i in range(len(eval_data))])
axes[1, 0].legend()
axes[1, 0].set_ylim(0, 5.5)
axes[1, 0].grid(True, alpha=0.3)

# 4. 개선율 히트맵
improvement_data = [
    [after - before for before, after in zip(bleu_before_scores, bleu_after_scores)],
    [after - before for before, after in zip(rouge_before_results['rouge1'], rouge_after_results['rouge1'])],
    [after - before for before, after in zip(bertscore_before, bertscore_after)],
    [after - before for before, after in zip(llm_overall_before, llm_overall_after)]
]

# 정규화 (각 메트릭의 스케일이 다르므로)
improvement_array = np.array(improvement_data)
im = axes[1, 1].imshow(improvement_array, cmap='RdYlGn', aspect='auto')
axes[1, 1].set_yticks(range(4))
axes[1, 1].set_yticklabels(['BLEU', 'ROUGE-1', 'BERTScore', 'LLM Judge'])
axes[1, 1].set_xticks(range(len(eval_data)))
axes[1, 1].set_xticklabels([f'Q{i+1}' for i in range(len(eval_data))])
axes[1, 1].set_title('Improvement Heatmap (After - Before)', fontweight='bold')
plt.colorbar(im, ax=axes[1, 1])

plt.suptitle('Fine-tuning Evaluation Results', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('evaluation_results.png', bbox_inches='tight', dpi=100)
plt.show()

print("✅ 종합 평가 결과 시각화 완료!")

In [ ]:
# 📊 평가 결과 요약 테이블
print("📊 파인튜닝 전후 평가 결과 요약")
print("=" * 70)

summary_data = {
    'Metric': ['BLEU', 'ROUGE-1', 'ROUGE-2', 'ROUGE-L', 'BERTScore', 'LLM Overall', 'LLM Accuracy', 'LLM Completeness'],
    'Before FT': [
        f"{np.mean(bleu_before_scores):.4f}",
        f"{np.mean(rouge_before_results['rouge1']):.4f}",
        f"{np.mean(rouge_before_results['rouge2']):.4f}",
        f"{np.mean(rouge_before_results['rougeL']):.4f}",
        f"{np.mean(bertscore_before):.4f}",
        f"{np.mean(llm_overall_before):.2f}",
        f"{np.mean(llm_accuracy_before):.2f}",
        f"{np.mean(llm_completeness_before):.2f}"
    ],
    'After FT': [
        f"{np.mean(bleu_after_scores):.4f}",
        f"{np.mean(rouge_after_results['rouge1']):.4f}",
        f"{np.mean(rouge_after_results['rouge2']):.4f}",
        f"{np.mean(rouge_after_results['rougeL']):.4f}",
        f"{np.mean(bertscore_after):.4f}",
        f"{np.mean(llm_overall_after):.2f}",
        f"{np.mean(llm_accuracy_after):.2f}",
        f"{np.mean(llm_completeness_after):.2f}"
    ]
}

df_summary = pd.DataFrame(summary_data)
print(df_summary.to_string(index=False))

print(f"\n📊 Pairwise 결과:")
if pairwise_results:
    print(f"   파인튜닝 후 승률: {wins_after/len(pairwise_results)*100:.1f}%")
else:
    print("   (Pairwise 평가가 수행되지 않았습니다.)")

---

## 7️⃣ 인간 평가 vs 자동 평가 상관관계

### 자동 평가의 한계

| 메트릭 | 인간 평가 상관관계 | 한계점 |
|--------|-----------------|--------|
| BLEU | 보통 (0.3~0.5) | 동의어/패러프레이즈 반영 안됨 |
| ROUGE | 보통 (0.3~0.5) | 단어 겹침만 측정 |
| BERTScore | 높음 (0.5~0.7) | 사실 정확성 미반영 |
| LLM Judge | 매우 높음 (0.7~0.9) | API 비용, 편향 가능성 |

### 인간 평가가 필요한 경우
- 🔹 **안전성 평가**: 유해 콘텐츠, 편향 감지
- 🔹 **도메인 전문성**: 의료/법률 등 전문 분야 정확성
- 🔹 **사용자 경험**: 대화 자연스러움, 톤 적절성
- 🔹 **최종 검증**: 배포 전 품질 보증

In [ ]:
# 📊 메트릭 간 상관관계 시각화
print("📊 메트릭 간 상관관계 분석")
print("=" * 60)

# 각 메트릭의 개선도 (after - before)
bleu_improvements = [a - b for a, b in zip(bleu_after_scores, bleu_before_scores)]
rouge1_improvements = [a - b for a, b in zip(rouge_after_results['rouge1'], rouge_before_results['rouge1'])]
bertscore_improvements = [a - b for a, b in zip(bertscore_after, bertscore_before)]
llm_improvements = [a - b for a, b in zip(llm_overall_after, llm_overall_before)]

# 상관관계 행렬
corr_data = np.array([
    bleu_improvements,
    rouge1_improvements,
    bertscore_improvements,
    llm_improvements
])

if corr_data.shape[1] > 1:  # 데이터가 2개 이상일 때만
    correlation_matrix = np.corrcoef(corr_data)
    
    fig, ax = plt.subplots(figsize=(8, 6))
    labels = ['BLEU', 'ROUGE-1', 'BERTScore', 'LLM Judge']
    
    im = ax.imshow(correlation_matrix, cmap='coolwarm', vmin=-1, vmax=1)
    ax.set_xticks(range(4))
    ax.set_yticks(range(4))
    ax.set_xticklabels(labels, rotation=45, ha='right')
    ax.set_yticklabels(labels)
    
    # 값 표시
    for i in range(4):
        for j in range(4):
            text = ax.text(j, i, f'{correlation_matrix[i, j]:.2f}',
                          ha='center', va='center', fontsize=12,
                          color='white' if abs(correlation_matrix[i, j]) > 0.5 else 'black')
    
    plt.colorbar(im)
    plt.title('Metric Correlation Matrix (Improvement)', fontweight='bold')
    plt.tight_layout()
    plt.savefig('metric_correlation.png', bbox_inches='tight', dpi=100)
    plt.show()

print("\n💡 LLM-as-a-Judge는 다른 자동 메트릭보다 인간 평가와 높은 상관관계를 보입니다.")
print("💡 하지만 완벽하지 않으므로, 중요한 결정에는 인간 평가를 병행해야 합니다.")

In [ ]:
# 📋 인간 평가 가이드라인 템플릿
print("📋 인간 평가 가이드라인 템플릿")
print("=" * 60)

human_eval_template = """
==========================================
  인간 평가 가이드라인 (Human Evaluation)
==========================================

평가자: ________________
날짜: ________________

[질문] {question}

[모델 응답] {response}

각 항목을 1~5점으로 평가해주세요.

1. 정확성 (Accuracy): [ ] / 5
   - 사실적으로 정확한 정보를 포함하는가?

2. 완전성 (Completeness): [ ] / 5
   - 질문에 대해 충분히 답변했는가?

3. 유창성 (Fluency): [ ] / 5
   - 문법이 올바르고 자연스러운가?

4. 유용성 (Helpfulness): [ ] / 5
   - 실제로 도움이 되는 답변인가?

5. 안전성 (Safety): [ ] / 5
   - 유해/편향적 내용이 없는가?

종합 점수: [ ] / 5

코멘트:
________________________________________
"""
print(human_eval_template)

print("💡 인간 평가는 최소 3명의 평가자로 구성하여 평가자 간 일치도(Inter-annotator Agreement)를 확인하세요.")
print("💡 Cohen's Kappa 또는 Fleiss' Kappa를 사용하여 일치도를 측정할 수 있습니다.")

---

## 8️⃣ 평가 데이터셋 구축 가이드

좋은 평가를 위해서는 **잘 구성된 평가 데이터셋**이 필수입니다.

### 평가 데이터셋 구축 원칙

- 🔹 **대표성**: 실제 사용 시나리오를 반영
- 🔹 **다양성**: 다양한 난이도, 주제, 형식 포함
- 🔹 **품질**: 참조 답변의 품질이 높아야 함
- 🔹 **규모**: 최소 50~100개 이상 (통계적 유의미성)
- 🔹 **균형**: 쉬운/어려운, 짧은/긴 질문 균형

In [ ]:
# 📝 평가 데이터셋 구축 자동화 코드
print("📝 평가 데이터셋 자동 구축 코드")
print("=" * 60)

eval_dataset_code = '''
from openai import OpenAI
import json

client = OpenAI()

def generate_eval_dataset(domain, num_questions=50):
    """도메인 특화 평가 데이터셋 자동 생성"""
    
    prompt = f"""당신은 {domain} 분야의 전문가입니다.
    
{domain} 분야에서 AI 모델을 평가하기 위한 질문-답변 쌍을 {num_questions}개 생성해주세요.

다음 규칙을 따르세요:
1. 질문은 다양한 난이도 (쉬움/보통/어려움)를 포함
2. 단답형, 설명형, 비교형 등 다양한 유형 포함
3. 참조 답변은 정확하고 완전해야 함
4. 실무에서 실제로 물어볼 만한 질문

JSON 배열 형식으로 출력:
[
    {{
        "id": 1,
        "question": "질문 내용",
        "reference_answer": "참조 답변",
        "difficulty": "easy/medium/hard",
        "category": "질문 카테고리"
    }},
    ...
]"""
    
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": "JSON 형식으로만 응답하세요."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.7,
        response_format={"type": "json_object"}
    )
    
    return json.loads(response.choices[0].message.content)

# 사용 예시
# eval_data = generate_eval_dataset("금융", num_questions=50)
# with open("eval_dataset_finance.json", "w") as f:
#     json.dump(eval_data, f, ensure_ascii=False, indent=2)
'''
print(eval_dataset_code)

print("\n💡 GPT-4를 사용하여 도메인 특화 평가 데이터셋을 자동 생성할 수 있습니다.")
print("💡 생성된 데이터셋은 도메인 전문가가 검수하는 것을 권장합니다.")

In [ ]:
# 📊 평가 파이프라인 통합 함수
print("📊 통합 평가 파이프라인")
print("=" * 60)

def run_full_evaluation(
    eval_dataset,
    predictions,
    references=None,
    use_llm_judge=True,
    judge_model="gpt-4o-mini"
):
    """통합 평가 파이프라인
    
    Args:
        eval_dataset: [{question, reference}, ...]
        predictions: [model_output, ...]
        references: [reference_answer, ...] (없으면 eval_dataset에서 추출)
        use_llm_judge: LLM 판정 사용 여부
        judge_model: 판정에 사용할 모델
    
    Returns:
        종합 평가 결과 딕셔너리
    """
    if references is None:
        references = [d['reference'] for d in eval_dataset]
    
    results = {
        'bleu_scores': [],
        'rouge_scores': [],
        'bertscore_f1': [],
        'llm_judge_scores': []
    }
    
    print("1️⃣ BLEU 계산 중...")
    smoother = SmoothingFunction().method1
    for ref, pred in zip(references, predictions):
        bleu = sentence_bleu([ref.split()], pred.split(), smoothing_function=smoother)
        results['bleu_scores'].append(bleu)
    
    print("2️⃣ ROUGE 계산 중...")
    rouge = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=False)
    for ref, pred in zip(references, predictions):
        scores = rouge.score(ref, pred)
        results['rouge_scores'].append({
            'rouge1': scores['rouge1'].fmeasure,
            'rouge2': scores['rouge2'].fmeasure,
            'rougeL': scores['rougeL'].fmeasure
        })
    
    print("3️⃣ BERTScore 계산 중...")
    try:
        from bert_score import score as bert_score_fn
        P, R, F1 = bert_score_fn(predictions, references, lang="ko", verbose=False)
        results['bertscore_f1'] = F1.numpy().tolist()
    except:
        results['bertscore_f1'] = [0.0] * len(predictions)
    
    if use_llm_judge:
        print("4️⃣ LLM-as-a-Judge 평가 중...")
        for d, pred in zip(eval_dataset, predictions):
            score = evaluate_with_llm_judge(
                question=d.get('question', ''),
                reference=d.get('reference', ''),
                response=pred,
                model=judge_model
            )
            results['llm_judge_scores'].append(score)
            time.sleep(0.5)
    
    # 요약 통계
    summary = {
        'avg_bleu': np.mean(results['bleu_scores']),
        'avg_rouge1': np.mean([r['rouge1'] for r in results['rouge_scores']]),
        'avg_rougeL': np.mean([r['rougeL'] for r in results['rouge_scores']]),
        'avg_bertscore': np.mean(results['bertscore_f1']),
    }
    
    if results['llm_judge_scores'] and results['llm_judge_scores'][0]:
        summary['avg_llm_overall'] = np.mean([
            s.get('overall', 0) for s in results['llm_judge_scores'] if s
        ])
    
    results['summary'] = summary
    print("\n✅ 평가 완료!")
    return results

print("✅ 통합 평가 파이프라인 함수 정의 완료!")
print("\n사용 예시:")
print('  results = run_full_evaluation(eval_dataset, predictions)')

In [ ]:
# 📊 평가 데이터셋 구축 체크리스트
print("📊 평가 데이터셋 구축 체크리스트")
print("=" * 60)

checklist = [
    ("데이터 수량", "최소 50~100개 이상의 평가 샘플", True),
    ("난이도 분포", "쉬움/보통/어려움 균형 (30/40/30)", True),
    ("질문 유형", "단답형, 설명형, 비교형, 추론형 등 다양한 유형", True),
    ("참조 답변", "전문가가 작성하거나 검수한 고품질 답변", True),
    ("도메인 범위", "실제 사용 시나리오를 반영한 질문", True),
    ("엣지 케이스", "비정상 입력, 모호한 질문 등 포함", False),
    ("다국어", "필요 시 한국어/영어 혼합", False),
    ("안전성", "유해 콘텐츠 거부 테스트 포함", False),
]

print("\n필수 항목:")
for name, desc, required in checklist:
    marker = "✅" if required else "📌"
    label = "[필수]" if required else "[권장]"
    print(f"  {marker} {label} {name}: {desc}")

print("\n💡 평가 데이터셋은 모델 학습에 사용되지 않은 데이터여야 합니다!")
print("💡 정기적으로 평가 데이터셋을 업데이트하여 모델 개선을 추적하세요.")

---

## 📝 정리 및 핵심 요약

### 이번 세션에서 배운 내용

1️⃣ **자동 평가 메트릭**: BLEU(정밀도), ROUGE(재현율), BERTScore(의미 유사도)

2️⃣ **메트릭 계산 실습**: 파인튜닝 전후 모델의 정량적 성능 비교

3️⃣ **LLM-as-a-Judge**: GPT-4를 활용한 자동 평가, 인간 평가와 높은 상관관계

4️⃣ **평가 프롬프트 설계**: Single Rating, Pairwise Comparison 방식

5️⃣ **자동 평가 실습**: OpenAI API로 파인튜닝 전후 모델 자동 평가

6️⃣ **결과 시각화**: matplotlib으로 다차원 평가 결과 분석

7️⃣ **인간 평가**: 자동 평가의 한계와 인간 평가의 필요성

8️⃣ **평가 데이터셋**: 체계적인 평가 데이터셋 구축 방법

### 핵심 키워드
- **LLM-as-a-Judge**: 가장 실용적인 자동 평가 방법
- **Pairwise Comparison**: 위치 편향 방지를 위한 랜덤 순서
- **JSON 출력 강제**: `response_format={"type": "json_object"}`
- **Temperature 0**: 일관된 평가를 위한 설정

### 평가 전략 요약
```
[개발 단계]     → 자동 메트릭 (BLEU, ROUGE, BERTScore)
[반복 개선]     → LLM-as-a-Judge (GPT-4)
[배포 전 검증]  → 인간 평가 + LLM-as-a-Judge
[프로덕션 모니터링] → 사용자 피드백 + 자동 평가
```

### 다음 세션 예고
- 🔜 **Part 7**: 프로젝트 - 도메인 특화 sLLM 구축

In [ ]:
# 🧹 최종 정리
print("\n" + "=" * 60)
print("✅ Session 32: 평가 메트릭, LLM-as-a-Judge 실습 완료!")
print("=" * 60)

print("\n📊 이번 세션에서 생성된 파일:")
import os
for f in ['evaluation_results.png', 'metric_correlation.png']:
    if os.path.exists(f):
        print(f"  ✅ {f}")

print("\n📌 Part 6: 배포 & 평가 전체 완료!")
print("\n🎓 주요 스킬 습득:")
print("  ✅ 양자화 (BnB, GPTQ, AWQ, GGUF)")
print("  ✅ API 서빙 (FastAPI + OpenAI 호환)")
print("  ✅ Streamlit 챗봇 구현")
print("  ✅ 자동 평가 메트릭 (BLEU, ROUGE, BERTScore)")
print("  ✅ LLM-as-a-Judge 자동 평가")
print("\n📌 다음 단계: Part 7 프로젝트 - 도메인 특화 sLLM 구축!")